<a href="https://colab.research.google.com/github/saparbayev-azizbek-12/bi-and-ai-talents-dl/blob/main/lesson-30/GPT2Model_Fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import tiktoken
import torch.nn as nn

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
cfg = {
    "vocab_size": tokenizer.n_vocab,
    "ctx_len": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": True
}

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_len, dropout, bias=False):
        super().__init__()

        if d_model % n_heads != 0:
            ValueError("d_model must be devidable to n_heads")
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.W_query = nn.Linear(d_model, d_model, bias=bias)
        self.W_key = nn.Linear(d_model, d_model, bias=bias)
        self.W_value = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_len, context_len), diagonal=1).bool()
        )
        self.output = nn.Linear(d_model, d_model, bias=bias)


    def forward(self, x):
        B, T, d_model = x.shape
        queries = self.W_query(x).view(B, T, self.n_heads, self.head_dim).transpose(1,2)
        keys = self.W_key(x).view(B, T, self.n_heads, self.head_dim).transpose(1,2)
        values = self.W_value(x).view(B, T, self.n_heads, self.head_dim).transpose(1,2)

        scores = queries @ keys.transpose(-1, -2)
        scores.masked_fill_(self.mask[:T, :T], -torch.inf)
        weight = torch.softmax(
            scores / (keys.size(-1)**0.5), dim=-1
        )
        context_vec = (weight @ values).transpose(1, 2).reshape(B, T, self.n_heads * self.head_dim)
        output = self.output(context_vec)
        return output


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        x_mean = x.mean(dim=-1, keepdim=True)
        x_var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm = (x - x_mean) / (torch.sqrt(x_var + self.eps))
        return self.scale * norm + self.shift


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim'], bias=cfg['qkv_bias']),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim'], bias=cfg['qkv_bias'])
        )

    def forward(self, x):
        return self.layers(x)


class Transformers(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_model=cfg['emb_dim'],
            n_heads=cfg['n_heads'],
            context_len=cfg['ctx_len'],
            dropout=cfg['drop_rate'],
            bias=cfg['qkv_bias']
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg['emb_dim'])
        self.norm2 = LayerNorm(cfg['emb_dim'])
        self.dropout = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.dropout(x)
        x = shortcut + x

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x = shortcut + x
        return x


class GPT2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['ctx_len'], cfg['emb_dim'])
        self.transformers_bloks = nn.Sequential(
            *[Transformers(cfg) for _ in range(cfg['n_layers'])]
        )
        self.dropout = nn.Dropout(cfg['drop_rate'])
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)
        self.out.weight = self.tok_emb.weight

    def forward(self, x):
        t_emb = self.tok_emb(x)
        p_emb = self.pos_emb(torch.arange(x.size(-1), device=x.device))
        x = t_emb + p_emb
        x = self.dropout(x)
        x = self.transformers_bloks(x)
        x = self.final_norm(x)
        x = self.out(x)
        return x

In [ ]:
my_model = GPT2Model(cfg)

In [ ]:
sum(p.numel() for p in my_model.parameters())

124439808

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

real_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
real_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
sum(p.numel() for p in real_model.parameters())

124439808

In [ ]:
my_model

GPT2Model(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (transformers_bloks): Sequential(
    (0): Transformers(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (output): Linear(in_features=768, out_features=768, bias=True)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='tanh')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): Transformers(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)


In [ ]:
real_model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
my_model.tok_emb.weight.data.copy_(real_model.transformer.wte.weight)
my_model.pos_emb.weight.data.copy_(real_model.transformer.wpe.weight)
my_model.final_norm.scale = real_model.transformer.ln_f.weight
my_model.final_norm.shift = real_model.transformer.ln_f.bias
my_model.out.weight.data.copy_(real_model.lm_head.weight)



tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]],
       grad_fn=<CopyBackwards>)

In [ ]:
for m, r in zip(my_model.transformers_bloks, real_model.transformer.h):
    c_attn_w = r.attn.c_attn.weight
    c_attn_b = r.attn.c_attn.bias

    m.att.W_query.weight.data.copy_(c_attn_w[:, :768].T)
    m.att.W_key.weight.data.copy_(c_attn_w[:, 768:1536].T)
    m.att.W_value.weight.data.copy_(c_attn_w[:, 1536:2304].T)

    m.att.W_query.bias.data.copy_(c_attn_b[:768])
    m.att.W_key.bias.data.copy_(c_attn_b[768:1536])
    m.att.W_value.bias.data.copy_(c_attn_b[1536:2304])

    m.att.output.weight.data.copy_(r.attn.c_proj.weight.T)
    m.att.output.bias.data.copy_(r.attn.c_proj.bias)

    m.norm1.scale.data.copy_(r.ln_1.weight)
    m.norm1.shift.data.copy_(r.ln_1.bias)

    m.norm2.scale.data.copy_(r.ln_2.weight)
    m.norm2.shift.data.copy_(r.ln_2.bias)

    m.ff.layers[0].weight.data.copy_(r.mlp.c_fc.weight.T)
    m.ff.layers[2].weight.data.copy_(r.mlp.c_proj.weight.T)

    m.ff.layers[0].bias.data.copy_(r.mlp.c_fc.bias)
    m.ff.layers[2].bias.data.copy_(r.mlp.c_proj.bias)


In [ ]:
prompt = 'What is the capital of Uzbekistan?'
my_token = torch.tensor(tokenizer.encode(prompt))
real_tokens = real_tokenizer(prompt, return_tensors='pt')

In [ ]:
output = real_model.generate(**real_tokens)
print(real_tokenizer.decode(output.squeeze()))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=28) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


What is the capital of Uzbekistan?

The capital of Uzbekistan is the capital of the Republic of Uzbekistan.

What


In [ ]:
@torch.no_grad()
def generate(model, tokens, max_new_tokens, context_size=cfg['ctx_len']):
    model.eval()
    tokens = tokens.unsqueeze(dim=0)
    for _ in range(max_new_tokens):
        token = tokens[:, -context_size:]
        logits = model(token)
        logits = logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        tokens = torch.cat((tokens, next_token), dim=1)

    return tokens

In [ ]:
out_probs = generate(my_model, my_token, 100)
print(tokenizer.decode(out_probs.squeeze().tolist()))

What is the capital of Uzbekistan?

The capital of Uzbekistan is the capital of the Republic of Uzbekistan.

What is the name of the country?

The capital of Uzbekistan is the capital of the Republic of Uzbekistan.

What is the name of the country?

The capital of Uzbekistan is the capital of the Republic of Uzbekistan.

What is the name of the country?

The capital of Uzbekistan is the capital of the Republic of Uzbekistan.




In [ ]:
my_model.out = nn.Linear(768, 4, bias=False)

In [ ]:
for param in my_model.parameters():
    param.requires_grad = False

for param in my_model.out.parameters():
    param.requires_grad = True
    print(param.requires_grad)

True


In [ ]:
from datasets import load_dataset

ds = load_dataset("sh0416/ag_news")

README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 33.7MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
ds['train'][0]

{'label': 3,
 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)',
 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}

In [ ]:
seed = 42

split_1 = ds['train'].train_test_split(test_size=0.2, seed=seed)
train_split = split_1['train']

split_2 = split_1['test'].train_test_split(test_size=0.5, seed=seed)
val_split = split_2['train']
test_split = split_2['test']

print(f"train: {len(train_split)}")
print(f"validation: {len(val_split)}")
print(f"test: {len(test_split)}")


train: 96000
validation: 12000
test: 12000


In [ ]:
from torch.utils.data import Dataset, DataLoader

PAD_TOKEN_ID = tokenizer.eot_token
class AGNewsDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=None, pad_token_id=PAD_TOKEN_ID):
        self.pad_token_id = pad_token_id
        self.encoded_texts = []
        self.labels = []

        for ex in data:
            text = ex['title'] + ": " + ex['description']
            self.encoded_texts.append(tokenizer.encode(text))
            self.labels.append(ex['label'])

        if max_length is None:
            self.max_length = min(max(len(text) for text in self.encoded_texts), cfg['ctx_len'])
        else:
            self.max_length = max_length

        for i, ids in enumerate(self.encoded_texts):
            ids = ids[:self.max_length]
            ids = ids + [self.pad_token_id] * (self.max_length - len(ids))
            self.encoded_texts[i] = ids

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.encoded_texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


In [ ]:
train_dataset = AGNewsDataset(train_split, tokenizer)
val_dataset = AGNewsDataset(val_split, tokenizer, max_length=train_dataset.max_length)
test_dataset = AGNewsDataset(test_split, tokenizer, max_length=train_dataset.max_length)

print("max_length:", train_dataset.max_length)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"train batches: {len(train_loader)}, val batches: {len(val_loader)}, test batches: {len(test_loader)}")


max_length: 343
train batches: 6000, val batches: 750, test batches: 750


In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    if len(data_loader) == 0:
        return float("nan")
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    total_loss = 0.0
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        loss = calc_loss_batch(input_batch, target_batch, model, device)
        total_loss += loss.item()
    return total_loss / num_batches


def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct, total = 0, 0
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    with torch.no_grad():
        for i, (input_batch, target_batch) in enumerate(data_loader):
            if i >= num_batches:
                break
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            logits = model(input_batch)[:, -1, :]
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == target_batch).sum().item()
            total += target_batch.size(0)
    model.train()
    return correct / total


## 4. Training loop (loss kuzatuvi bilan)

Har `eval_freq` step sayin train/validation/test loss hisoblanib, tarixga yoziladi (keyin grafik chizish uchun), har epoch oxirida esa to'liq accuracy chiqariladi.

In [ ]:
def evaluate_model(model, train_loader, val_loader, test_loader, device, eval_iter=5):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
        test_loss = calc_loss_loader(test_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss, test_loss


def train_classifier(model, train_loader, val_loader, test_loader, optimizer, device,
                      num_epochs, eval_freq=50, eval_iter=5):
    train_losses, val_losses, test_losses, track_steps = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.size(0)
            global_step += 1

            if global_step % eval_freq == 0:
                tr_loss, v_loss, te_loss = evaluate_model(
                    model, train_loader, val_loader, test_loader, device, eval_iter
                )
                train_losses.append(tr_loss)
                val_losses.append(v_loss)
                test_losses.append(te_loss)
                track_steps.append(global_step)
                print(f"Epoch {epoch+1} | Step {global_step:06d} | "
                      f"Train loss {tr_loss:.3f} | Val loss {v_loss:.3f} | Test loss {te_loss:.3f}")

        train_acc = calc_accuracy_loader(train_loader, model, device, num_batches=20)
        val_acc = calc_accuracy_loader(val_loader, model, device, num_batches=20)
        test_acc = calc_accuracy_loader(test_loader, model, device, num_batches=20)
        print(f"--- Epoch {epoch+1} tugadi | Train acc {train_acc*100:.2f}% | "
              f"Val acc {val_acc*100:.2f}% | Test acc {test_acc*100:.2f}% ---")

    return train_losses, val_losses, test_losses, track_steps, examples_seen


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
my_model.to(device)
print("device:", device)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, my_model.parameters()),
    lr=5e-4, weight_decay=0.01
)

num_epochs = 3
train_losses, val_losses, test_losses, track_steps, examples_seen = train_classifier(
    my_model, train_loader, val_loader, test_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5
)


device: cuda
Epoch 1 | Step 000000 | Train loss 2.799 | Val loss 3.257 | Test loss 3.320
Epoch 1 | Step 000050 | Train loss 1.411 | Val loss 1.400 | Test loss 1.395
Epoch 1 | Step 000100 | Train loss 1.538 | Val loss 1.550 | Test loss 1.531
Epoch 1 | Step 000150 | Train loss 1.405 | Val loss 1.398 | Test loss 1.382
Epoch 1 | Step 000200 | Train loss 1.422 | Val loss 1.371 | Test loss 1.368
Epoch 1 | Step 000250 | Train loss 1.417 | Val loss 1.385 | Test loss 1.390
Epoch 1 | Step 000300 | Train loss 1.382 | Val loss 1.365 | Test loss 1.359
Epoch 1 | Step 000350 | Train loss 1.365 | Val loss 1.379 | Test loss 1.363
Epoch 1 | Step 000400 | Train loss 1.345 | Val loss 1.427 | Test loss 1.404
Epoch 1 | Step 000450 | Train loss 1.378 | Val loss 1.393 | Test loss 1.377
Epoch 1 | Step 000500 | Train loss 1.366 | Val loss 1.359 | Test loss 1.357
Epoch 1 | Step 000550 | Train loss 1.360 | Val loss 1.369 | Test loss 1.355
Epoch 1 | Step 000600 | Train loss 1.376 | Val loss 1.363 | Test loss 1.351

KeyboardInterrupt: 